# Data Anonymisation Techniques

## ⚠️ Important: Educational Exercise: Do Not Use on Real Data

This notebook uses anonymisation techniques to help you explore how data can be made less identifiable. These examples are for **learning purposes only**. They are **not suitable for real datasets** unless additional safeguards are in place.

---

### ❗ Key limitations of these techniques

- **Individuals can still be identified**, especially when data is combined with other sources  
- **Results depend on the context**, what works for one dataset or use case may fail in another  
- **This is not production-ready**, it’s a simplified example, not a complete privacy solution

---

### ✅ If you're working with real personal data:

- **Always consult your Data Protection Officer (DPO)** or Information Governance team  
- **Carry out a risk assessment** before sharing or using the data  
- **Consider your use case**, public release and internal analysis require different levels of protection  
- **Explore other options**, synthetic data may be safer for many development tasks

---

**🔐 Remember:** Anonymisation is not a one-size-fits-all solution. When in doubt, don’t guess, ask!


The examples and functions used in the original version of this workbook, and in the task solution videos, utilise the Anonympy library for Python. There are other Python libraries which can achieve the same results using different functions and methods. The purpose of this workbook is to demonstrate a range of data anonymisation techniques which are not specific to a particular Python library.

> 📗 **About this browser version:** Anonympy cannot be installed in the browser, so this notebook carries out each step with **pandas** and **Faker** (the library Anonympy itself uses to generate fake data). The steps, the columns and the technique names are the same as in the videos, and each code cell shows the Anonympy line it replaces as a comment. The code looks different from the videos, but each step does the same thing to the data.

Faker is not built into the browser's Python, so install it first by running the cell below.

In [ ]:
%pip install -q faker

# The original notebook installed Anonympy here:
# !pip install git+https://github.com/ArtLabss/open-data-anonymizer.git
# !pip install cape-privacy --no-deps

## Importing and viewing our data

Lets take a look at how we can anonymise data within Python.
We will start by importing the Pandas library for our dataframe, and the Faker library to generate fake data. (The original imported the Anonympy library here.) The source data we are using is randomly generated dummy data and is specific to this workbook example.

In [ ]:
import pandas as pd #Import Pandas
import numpy as np #Import NumPy, for random noise
import hashlib, secrets #Python's built-in hashing and random key tools, for tokenisation
from faker import Faker #Import Faker, for synthetic data
# Original: from anonympy.pandas import dfAnonymizer #Import Anonympy

We will use Pandas to import some fictional customer data from the file `customer_data.csv` in this lab to work with and create a Pandas dataframe object called `customer_data`. Because our customer data contains dates, we must parse the data on the column containing our dates `date_of_birth`.

In [ ]:
# Import the CSV file into a Pandas DataFrame
# (originally read from https://raw.githubusercontent.com/Corndel/corndel-datasets/refs/heads/main/synthetic/customer_data.csv)
customer_data = pd.read_csv(
    'customer_data.csv',
    parse_dates=['date_of_birth'],
    dayfirst=True
)

# Display the first few rows of the DataFrame
customer_data.head()

We have used the `.head()` function of pandas to view the first 5 rows of our `customer_data` dataframe object. This includes the headers for each column. This is useful and reveals information which is useful to us, including the type of data found in our dataframe.

## Anonymising the data

We begin by creating a copy of our `customer_data` dataframe called `anon_data`, which we will anonymise, and a dictionary called `methods_applied` to record the technique we use on each column. (With Anonympy, this was a `dfAnonymizer` object.) It is useful at this stage to see which of our columns fall into which category of data, as this will affect the type of anonymisation technique we need to use. We can find this out by using the `print` function and pandas' `.select_dtypes()` function to list the numeric, categorical (text) and date columns.

In [ ]:
anon_data = customer_data.copy() # Create a copy of our customer_data called anon_data to anonymise
methods_applied = {} # Record which anonymisation technique we apply to each column

numeric_columns = anon_data.select_dtypes(include='number').columns.tolist()
categorical_columns = anon_data.select_dtypes(include=['object', 'string']).columns.tolist()
datetime_columns = anon_data.select_dtypes(include='datetime').columns.tolist()

print('Numeric columns', numeric_columns) # Print columns containing numerical data
print('Categorical columns', categorical_columns) # Print columns containing categorical data
print('Date columns', datetime_columns) # Print columns containing dates

# Original (Anonympy):
# anon_data = dfAnonymizer(customer_data)
# print('Numeric columns', anon_data.numeric_columns)
# print('Categorical columns', anon_data.categorical_columns)
# print('Date columns', anon_data.datetime_columns)

This has revealed the column data types and will inform the anonymisation functions we need to select. In this example, you can see we have 1 numeric column, 7 categorical columns and 1 date column.

## Pseudonymisation

### ⚠️ Note on Pseudonymization Limitations

**Pseudonymization replaces identifiers with artificial ones, but important limitations exist:**

- **Not true anonymization**: Pseudonymized data can often be re-identified if the mapping between original and pseudonymized identifiers is discovered
- **Linkage risks**: When combined with other datasets, pseudonymized records may still be identifiable
- **Context matters**: The effectiveness depends on what other data is available and how the pseudonyms are generated

**In practice**: Always assess whether pseudonymization provides sufficient protection for your specific use case and data environment.

Lets apply some pseudonymisation to our `first_name` and `last_name` columns. We will use Faker's `.first_name()` and `.last_name()` functions to replace each name in our data with a fictitious one (Anonympy's `.categorical_fake()` function does the same, using Faker). We can then display `anon_data` to see the results.

In [ ]:
fake = Faker('en_US') # A Faker generator. Anonympy uses the same 'en_US' locale by default

anon_data['first_name'] = anon_data['first_name'].apply(lambda x: fake.first_name()) # Replace each first name with a fake one
anon_data['last_name'] = anon_data['last_name'].apply(lambda x: fake.last_name()) # Replace each last name with a fake one
methods_applied['first_name'] = 'Synthetic Data'
methods_applied['last_name'] = 'Synthetic Data'

anon_data # Display the results of our anonymisation

# Original (Anonympy):
# anon_data.categorical_fake('first_name')
# anon_data.categorical_fake('last_name')
# anon_data.to_df()

You should see that the original names in our dataset have been replaced with pseudonyms, or ficticious names.

## Masking

### ⚠️ Note on Masking Limitations

**Masking partially hides data but retains some information for utility:**

- **Partial information leakage**: Masked data still reveals patterns (e.g., email domains, partial names)
- **Inference risks**: Attackers may be able to infer the hidden information from the visible parts
- **Consistency issues**: The same masking pattern applied across datasets can create linkage opportunities

Lets take a look at how we can apply masking techniques to disguise the email addresses. We will write a small function, `mask_email()`, that keeps the first character of the address and everything from the character before the `@`, and replaces the rest with `*****`. This is the same rule Anonympy's `.categorical_email_masking()` function applies. We then apply it to the `email_address` column and display the results.

In [ ]:
def mask_email(email):
    at = email.find('@') # Position of the @ sign
    return email[0] + '*****' + email[at - 1:] # Keep the first character and the end, mask the middle

anon_data['email_address'] = anon_data['email_address'].apply(mask_email) # Apply mask_email to the email_address column
methods_applied['email_address'] = 'Partial Masking'

anon_data # Display the results of our anonymisation

# Original (Anonympy):
# anon_data.categorical_email_masking('email_address')
# anon_data.to_df()

This method preserves details such as the email domain (@gmail.com,@yahoo.com etc), but masks the details of the first part of the email address.

## Tokenisation

### ⚠️ Note on Tokenisation Limitations

**Tokenisation replaces sensitive data with random tokens, but risks remain:**

- **Token-data mapping**: If the tokenisation system is compromised, original data can be recovered
- **Consistency patterns**: Using the same tokenisation across multiple datasets can create linkage opportunities
- **Statistical analysis**: Large datasets may still reveal patterns even with tokenised data

We can tokenise our data by replacing each value with a token: a hash of the value combined with a secret random key, cut to 10 characters. The same value always gets the same token, but without the key the token cannot be turned back into the value. This is how Anonympy's `.categorical_tokenization()` function works. We will apply this to both the `street_address` and `post_code` columns.

In [ ]:
key = secrets.token_bytes(8) # A secret random key. Keep it secret: anyone with the key can test guesses against the tokens

def tokenise(value, max_token_len=10):
    return hashlib.sha256(value.encode() + key).hexdigest()[:max_token_len] # Hash the value with the key, keep 10 characters

anon_data['street_address'] = anon_data['street_address'].apply(tokenise) # Tokenise the street_address column
anon_data['post_code'] = anon_data['post_code'].apply(tokenise) # Tokenise the post_code column
methods_applied['street_address'] = 'Tokenization'
methods_applied['post_code'] = 'Tokenization'

anon_data # Display the results of our anonymisation

# Original (Anonympy):
# anon_data.categorical_tokenization('street_address')
# anon_data.categorical_tokenization('post_code')
# anon_data.to_df()

We can see that both the `street_address` and `post_code` columns have been tokenised with randomised keys made up of alphanumerical characters.

## Perturbation (Numeric Noise)

### ⚠️ Note on Perturbation Limitations

**Perturbation (adding noise) can protect individual values but has important limitations:**

- **Utility trade-off**: More noise increases privacy but reduces data utility for analysis
- **Statistical attacks**: Sophisticated attackers may be able to filter out noise with enough data points
- **Consistency issues**: Related data points (like age and date of birth) can create contradictions that reveal the original values

So far, we have applied categorical anonymisation techniques. Lets have a look at how to add some perturbation (numeric noise) to the `age` column. We will add a random whole number between -10 and 10 to each age, which is what Anonympy's `.numeric_noise()` function does by default.

In [ ]:
rng = np.random.default_rng() # A random number generator
noise = rng.uniform(-10, 10, size=len(anon_data)).astype(int) # Random noise between -10 and 10, as whole numbers
anon_data['age'] = anon_data['age'] + noise # Apply numeric noise to the age column
methods_applied['age'] = 'Numeric Perturbation'

anon_data # Display the results of our anonymisation

# Original (Anonympy):
# anon_data.numeric_noise('age')
# anon_data.to_df()

This has adjusted the age values slightly in the `age` column. However, it would still be possible to recalculate this value from the `date_of_birth` column. We could either apply perturbation (random noise) to our `date_of_birth` column. Or we could hide this column completely from our data by dropping it, which is called column suppression (Anonympy's `.column_suppression()` function).

In [ ]:
anon_data = anon_data.drop(columns='date_of_birth') # Apply column suppression to the date_of_birth column
methods_applied['date_of_birth'] = 'Column Suppression'

anon_data # Display the results of our anonymisation

# Original (Anonympy):
# anon_data.column_suppression('date_of_birth')
# anon_data.to_df()

## Lets Summarise our Anonymisation

We can gather information about the changes we have made to our data by building a summary from the `methods_applied` dictionary. This will create a summary of what anonymisation techniques we have used on which columns, in the same layout as Anonympy's `.info()` function: `Status` is 1 for a column we have anonymised and 0 for one we have not.

In [ ]:
summary = pd.DataFrame({
    'Column': customer_data.columns,
    'Status': [1 if col in methods_applied else 0 for col in customer_data.columns],
    'Type': ['numeric' if col in numeric_columns else 'categorical' if col in categorical_columns else 'datetime'
             for col in customer_data.columns],
    'Method': [methods_applied.get(col, '') for col in customer_data.columns],
})
summary # Display summary information for the anonymisation we have undertaken

# Original (Anonympy):
# anon_data.info()

You should be able to see from the summary information, that we have generated synthetic data for `first_name` and `last_name` columns. Applied column suppression to `date_of_birth`, numeric perturbation to `age`, partial masking to `email_address` and tokenisation to `street_address` and `post_code`.